# regkit demo

Quick smoke-test for the regression benchmark pipeline.

Give it a **preprocessed** DataFrame; it tunes several models for **R²** and prints a leaderboard plus the winning hyperparameters.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from sklearn.datasets import make_regression

from regkit import benchmark

In [2]:
X, y = make_regression(
    n_samples=400,
    n_features=12,
    n_informative=8,
    noise=10.0,
    random_state=42,
)

df = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
df["target"] = y
df.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,target
0,-0.225656,-2.832326,-1.220321,-2.017339,-0.679231,-2.914256,0.837764,1.389024,0.386383,0.813981,0.523223,0.128617,-275.866377
1,-1.292625,-0.147025,-0.832061,-0.550115,-0.674934,-0.403076,2.046862,-0.532169,-0.222475,0.840734,1.991553,0.207803,103.468253
2,-1.415267,-1.406815,-0.322028,1.325432,1.669070,-0.942558,-0.130060,0.218534,-0.556119,0.585904,1.614779,-0.736770,102.009410
3,-0.340599,0.695864,-0.561558,-0.553090,-1.781158,-2.230468,-0.482828,0.500143,0.102485,1.173442,0.226684,-0.272545,-233.233719
4,0.141257,1.388338,-0.499730,-0.526248,-0.606865,0.457687,-2.120700,-2.130566,-2.238231,-0.385022,-2.747505,0.382989,-405.876789


In [3]:
# Smaller model set + fewer CV folds for a fast demo.
# In the contest, drop `models=` to run the full GridSearch suite.
result = benchmark(
    df,
    target="target",
    cv=3,
    n_jobs=-1,
    models=[
        "Ridge",
        "RandomForestRegressor",
        "XGBRegressor",
        "LGBMRegressor",
        "CatBoostRegressor",
    ],
)

result

BenchmarkResult
--------------
best_model : Ridge
best_score : 0.995234  (R²)
best_params: {'alpha': 0.1, 'solver': 'auto'}

                model  mean_r2   std_r2                                                                                             best_params  fit_time_sec status
                Ridge 0.995234 0.000318                                                                        {'alpha': 0.1, 'solver': 'auto'}         1.836     ok
    CatBoostRegressor 0.946173 0.004231                                 {'depth': 4, 'iterations': 400, 'l2_leaf_reg': 3, 'learning_rate': 0.1}        21.791     ok
         XGBRegressor 0.904442 0.005810 {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.8}         4.945     ok
        LGBMRegressor 0.885754 0.025303        {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 400, 'num_leaves': 31, 'subsample': 0.8}         0.998     ok
RandomForestRegressor 0.767230 0.012816           

In [4]:
print("best_model :", result.best_model)
print("best_score :", result.best_score)
print("best_params:", result.best_params)
result.leaderboard

best_model : Ridge
best_score : 0.9952342964370805
best_params: {'alpha': 0.1, 'solver': 'auto'}


,model,mean_r2,std_r2,best_params,fit_time_sec,status
0,Ridge,0.995234,0.000318,"{'alpha': 0.1, 'solver': 'auto'}",1.836,ok
1,CatBoostRegressor,0.946173,0.004231,"{'depth': 4, 'iterations': 400, 'l2_leaf_reg':...",21.791,ok
2,XGBRegressor,0.904442,0.005810,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",4.945,ok
3,LGBMRegressor,0.885754,0.025303,"{'learning_rate': 0.05, 'max_depth': 5, 'n_est...",0.998,ok
4,RandomForestRegressor,0.767230,0.012816,"{'max_depth': None, 'max_features': 1.0, 'min_...",3.356,ok
